# AECCT on LDPC(49,24)

This notebook clones the repository, checks the Kaggle GPU, trains the AECCT configuration used for comparison, and prints the resulting BER/FER log.

Before running it, push `AECCT-main/` to the GitHub repository. The current local folder is untracked, so it will not be included by `git clone` until you commit and push it. Enable **Internet** and a **GPU accelerator** in Kaggle.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/gouravanirudh05/SRIP_LDPC_Decoding_using_Machine_Learning.git'
REPO_DIR = Path('/kaggle/working/ldpc_repo')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

AECCT_DIR = REPO_DIR / 'AECCT-main'
if not (AECCT_DIR / 'main.py').exists():
    raise FileNotFoundError('AECCT-main is missing from the cloned repository. Commit and push it first.')

os.chdir(AECCT_DIR)
print('Working directory:', Path.cwd())

In [ ]:
# Kaggle normally already provides PyTorch. Install the packages required by AECCT.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'tqdm'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. In Kaggle, select GPU under Notebook options.')
print('GPU:', torch.cuda.get_device_name(0))

## Training configuration

AECCT uses 6 transformer blocks and dimension 128 on `LDPC_N49_K24`. This faster comparison run performs 500 normal-training epochs followed by 500 quantization-aware-training epochs.

In [ ]:
import time

command = [
    sys.executable, 'main.py',
    '--code', 'LDPC_N49_K24',
    '--N_dec', '6',
    '--d_model', '128',
    '--epochs', '500',
    '--workers', '4',
    '--batch_size', '128',
    '--test_batch_size', '2048',
    '--lr', '1e-4',
    '--seed', '42',
]
print('Running:', ' '.join(command))
start = time.perf_counter()
subprocess.run(command, cwd=str(AECCT_DIR), check=True)
print(f'Total wall time: {(time.perf_counter() - start) / 3600:.2f} hours')

In [ ]:
# Print the latest AECCT result directory and its final log lines.
result_dirs = sorted((AECCT_DIR / 'logs' / 'Results_AECCT').glob('*'), key=lambda p: p.stat().st_mtime)
if not result_dirs:
    raise FileNotFoundError('No AECCT result directory was produced.')
latest = result_dirs[-1]
log_file = latest / 'logging.txt'
print('Result directory:', latest)
print('Checkpoint:', latest / 'best_model')
print('\n'.join(log_file.read_text(errors='replace').splitlines()[-40:]))

In [ ]:
from pathlib import Path
import tarfile

# For ECCT:
result_root = Path("/kaggle/working/ldpc_repo/ECCT/Results_ECCT")

# For AECCT, use instead:
# result_root = Path("/kaggle/working/ldpc_repo/AECCT-main/logs/Results_AECCT")

result_dirs = sorted(result_root.glob("*"), key=lambda p: p.stat().st_mtime)
latest = result_dirs[-1]

archive = Path("/kaggle/working/LDPC49_results.tar.gz")

with tarfile.open(archive, "w:gz") as tar:
    tar.add(latest, arcname=latest.name)

print("Saved:", archive)
print("Log:", latest / "logging.txt")
print("Checkpoint:", latest / "best_model")